# Operational Benchmark Estimation

This notebook documents the active estimation benchmark layer only. Historical enrollment-only and site-only benchmark artifacts, builders, checkers, and runtime utilities have been removed. The active runtime source is `src/operational_benchmarks.py`, with one builder and one checker:

```bash
python scripts/build_operational_benchmarks.py
python scripts/check_operational_benchmarks.py
```

The production UI reads `frontend/data/operational_benchmarks_v1.csv` for Planned Enrollment, Planned Site Count, and Duration metadata/defaulting in Simulation Mode. The Excel companion `frontend/data/operational_benchmarks_v1.xlsx` is for analyst inspection only.

## Operational Benchmark Rules Reference

The active estimation layer uses one combined artifact: `frontend/data/operational_benchmarks_v1.csv`, built by `scripts/build_operational_benchmarks.py` and read at runtime by `src/operational_benchmarks.py`. The app also ships `operational_benchmarks_v1_report.json` for build summary and `operational_benchmarks_v1.xlsx` for human inspection only.

**Source populations**

- Enrollment percentiles use completed trials with positive `ACTUAL` enrollment.
- Site-count percentiles use completed trials with positive `number_of_facilities`.
- Patients-per-site percentiles use completed trials with positive `ACTUAL` enrollment and positive `number_of_facilities`, with `patients_per_site = enrollment / number_of_facilities`.
- Primary-completion percentiles use completed trials with `primary_completion_date_type = ACTUAL` and positive `primary_completion_duration_months`.
- Total-duration percentiles use completed trials with `completion_date_type = ACTUAL` and positive `completion_duration_months`.
- `number_of_facilities` is a registry-derived facility-count proxy, not true planned sites or true activated sites.
- Percentiles are deterministic historical benchmarks, not an ML model, and do not enter XGBoost, SHAP, `/predict`, Completion Score, calibration, taxonomy, or prediction payloads.

**Cohort hierarchy**

Runtime lookup tries the strongest valid clinical cohort in this order:

1. `phase_indication_rare`: phase + indication + rare flag
2. `phase_ta_rare`: phase + therapeutic area + rare flag
3. `phase_ta`: phase + therapeutic area
4. `phase_only`: phase only

Duration metrics first try endpoint-duration-bin variants of the same hierarchy:

1. `phase_indication_rare_endpoint_bin`
2. `phase_ta_rare_endpoint_bin`
3. `phase_ta_endpoint_bin`
4. `phase_endpoint_bin`

If `primary_duration_months_ml` cannot form a valid endpoint bin, duration falls back to the clinical hierarchy without the bin.

Invalid placeholder values cannot become specific cohorts:

- Indication id `0` or missing disables indication-level cohorts only.
- TA values `OTHER/UNCLASSIFIED`, `UNCLASSIFIED`, `UNKNOWN`, `OTHER`, or missing disable TA-level cohorts only.
- If indication is valid but TA is invalid, indication-level lookup is still allowed.
- Unknown/unclassified modality never creates or selects a modality-refinement row.

**Support thresholds**

- `n >= 50`: confident evidence.
- `30 <= n < 50`: usable low-confidence evidence for enrollment/site clinical fallback rows.
- `n < 30`: too sparse for that metric; fallback is required when possible.
- A selected enrollment/site clinical row is kept if at least one relevant operational metric is confident; weaker metrics on the same row remain flagged low confidence.
- Duration is stricter: editable total duration requires `duration_months_n >= 50`; primary-completion context is read from the same selected full-duration row only when `primary_completion_months_n >= 50`.
- Modality refinement and non-vaccine Infections fallback require `n >= 50` for the refined metric.

**Modality and Infections rules**

- Same-level modality refinement applies only to `enrollment` and `patients_per_site`, never raw `site_count` or duration.
- Modality refinement is same-level only: it refines the already selected clinical level and never jumps to `phase + modality`.
- Non-vaccine Infections fallback applies only to `enrollment` and `patients_per_site`, never raw `site_count` or duration.



In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "frontend" / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
ARTIFACT_PATH = PROJECT_ROOT / "frontend" / "data" / "operational_benchmarks_v1.csv"
REPORT_PATH = PROJECT_ROOT / "frontend" / "data" / "operational_benchmarks_v1_report.json"
EXCEL_PATH = PROJECT_ROOT / "frontend" / "data" / "operational_benchmarks_v1.xlsx"

artifact = pd.read_csv(ARTIFACT_PATH)
artifact.shape

(7406, 43)

## Artifact Structure

Each row is a benchmark cohort. Cohort keys describe the matching level: phase, indication, therapeutic area, rare flag, optional modality/non-vaccine Infections refinement, and optional endpoint-duration bin for duration metrics. Metric groups contain sample size and percentiles for:

- `enrollment_*`
- `site_count_*`
- `patients_per_site_*`
- `primary_completion_months_*`
- `duration_months_*`

The runtime uses `p25`, `p50`, `p75`, and `p90` to classify operational assumptions as below benchmark, typical, ambitious, or above high benchmark.


In [2]:
artifact.columns.tolist()

['benchmark_version',
 'source_data_version',
 'benchmark_key',
 'phase',
 'gbd_cause_id_3_ml',
 'therapeutic_area',
 'rare_disease_flag',
 'therapeutic_modality',
 'endpoint_duration_bin',
 'benchmark_level_used',
 'enrollment_n',
 'enrollment_p25',
 'enrollment_p50',
 'enrollment_p75',
 'enrollment_p90',
 'enrollment_low_confidence_flag',
 'site_count_n',
 'site_count_p25',
 'site_count_p50',
 'site_count_p75',
 'site_count_p90',
 'site_count_low_confidence_flag',
 'patients_per_site_n',
 'patients_per_site_p25',
 'patients_per_site_p50',
 'patients_per_site_p75',
 'patients_per_site_p90',
 'patients_per_site_low_confidence_flag',
 'primary_completion_months_n',
 'primary_completion_months_p25',
 'primary_completion_months_p50',
 'primary_completion_months_p75',
 'primary_completion_months_p90',
 'primary_completion_months_low_confidence_flag',
 'duration_months_n',
 'duration_months_p25',
 'duration_months_p50',
 'duration_months_p75',
 'duration_months_p90',
 'duration_months_low_c

In [3]:
artifact['benchmark_level_used'].value_counts().sort_index()

benchmark_level_used
phase_endpoint_bin                                32
phase_indication_rare                            656
phase_indication_rare_endpoint_bin              2232
phase_indication_rare_modality                  1948
phase_indication_rare_non_vaccine_infections      81
phase_only                                         4
phase_ta                                          72
phase_ta_endpoint_bin                            438
phase_ta_modality                                440
phase_ta_non_vaccine_infections                    4
phase_ta_rare                                    133
phase_ta_rare_endpoint_bin                       698
phase_ta_rare_modality                           660
phase_ta_rare_non_vaccine_infections               8
Name: count, dtype: int64

## Runtime Lookup Contract

The runtime first tries clinical specificity:

1. `phase_indication_rare`
2. `phase_ta_rare`
3. `phase_ta`
4. `phase_only`

Duration metrics try endpoint-duration-bin versions of those cohorts before falling back to the clinical-only hierarchy. Invalid indication disables only indication-level cohorts. Invalid/unclassified TA disables only TA-level cohorts. Unknown/unclassified modality cannot form a modality-refinement row.

For enrollment and patients-per-site only, the runtime may refine the selected clinical row with same-level modality when `n >= 50`. For non-vaccine Infections, if modality refinement is unavailable, it may use Infections rows excluding vaccines, also requiring `n >= 50`. Raw site-count and duration remain clinical-only apart from duration endpoint-bin matching.


In [4]:
import importlib
import src.operational_benchmarks as opb

opb = importlib.reload(opb)
benchmarks = opb.load_operational_benchmarks(ARTIFACT_PATH)
example_snapshot = {
    "phase": "PHASE3",
    "gbd_cause_id_3_ml": 302,
    "therapeutic_area": "INFECTIONS",
    "is_rare_disease_ml": 0,
    "therapeutic_modality_ui": "VACCINE",
    "primary_duration_months_ml": 9,
}

{
    metric: opb.lookup_operational_benchmark(example_snapshot, benchmarks, metric_prefix=metric)[
        ["benchmark_level_used", f"{metric}_n", f"{metric}_p50", f"{metric}_low_confidence_flag"]
    ].to_dict()
    for metric in ("enrollment", "site_count", "patients_per_site", "primary_completion_months", "duration_months")
}


{'enrollment': {'benchmark_level_used': 'phase_ta_rare_modality',
  'enrollment_n': 691,
  'enrollment_p50': 750.0,
  'enrollment_low_confidence_flag': False},
 'site_count': {'benchmark_level_used': 'phase_ta_rare',
  'site_count_n': 1003,
  'site_count_p50': 14.0,
  'site_count_low_confidence_flag': False},
 'patients_per_site': {'benchmark_level_used': 'phase_ta_rare_modality',
  'patients_per_site_n': 647,
  'patients_per_site_p50': 75.0,
  'patients_per_site_low_confidence_flag': False},
 'primary_completion_months': {'benchmark_level_used': 'phase_ta_rare_endpoint_bin',
  'primary_completion_months_n': 160,
  'primary_completion_months_p50': 17.82,
  'primary_completion_months_low_confidence_flag': False},
 'duration_months': {'benchmark_level_used': 'phase_ta_rare_endpoint_bin',
  'duration_months_n': 159,
  'duration_months_p50': 30.03,
  'duration_months_low_confidence_flag': False}}

## Defaulting Rules

Planned Enrollment uses a planned/estimated value when available. For non-completed trials without a planned/estimated value, current observed enrollment is treated as a lower bound and the default is:

```text
max(observed_lower_bound, enrollment_p50)
```

Planned Sites uses completed registry facility count for completed trials. For non-completed trials, current registry facility count is lower-bound context and the default is:

```text
max(current_registry_facility_count_proxy, planned_enrollment / patients_per_site_p50)
```

Pure `site_count_p50` is fallback/reference when patients-per-site cannot be calculated.

Planned Duration uses total operational duration from `start_date` to `completion_date`. The editable duration benchmark is selected only from cohorts with `duration_months_n >= 50`. Primary-completion timing is non-editable readout context from that same selected full-duration row when `primary_completion_months_n >= 50`.

Completed `ACTUAL` completion durations and active/non-stopped `ESTIMATED` completion durations are direct trusted values. Stopped/interrupted actual or estimated dates are lower-bound/floor context, so the fallback default is:

```text
max(duration_p50, stopped_total_duration_floor)
```


In [5]:
opb.planned_enrollment_default_from_operational_benchmark(
    example_snapshot,
    observed_lower_bound=150,
    artifact=benchmarks,
)

{'value': 750,
 'source': 'model_default',
 'observed_lower_bound': 150.0,
 'enrollment_benchmark_p50': 750.0,
 'operational_benchmark_snapshot_id': 'operational_benchmarks_v1:41bcafd6226cd999:phase_ta_rare_modality|phase=PHASE3|ta=INFECTIONS|rare=0|modality=VACCINE'}

In [6]:
opb.planned_sites_default_from_operational_benchmark(
    example_snapshot,
    planned_enrollment=500,
    current_registry_facility_count_proxy=12,
    overall_status="RECRUITING",
    artifact=benchmarks,
)

{'value': 12,
 'source': 'current_registry_facility_count_proxy',
 'site_default_basis': 'current_registry_facility_count_proxy',
 'current_registry_facility_count_proxy': 12.0,
 'site_count_benchmark_p50': 14.0,
 'patients_per_site_p50': 75.0,
 'patients_per_site_benchmark_level_used': 'phase_ta_rare_modality',
 'patients_per_site_n': 647,
 'patients_per_site_low_confidence_flag': False,
 'enrollment_coherent_site_candidate': 6.666666666666667,
 'operational_benchmark_snapshot_id': 'operational_benchmarks_v1:41bcafd6226cd999:phase_ta_rare_modality|phase=PHASE3|ta=INFECTIONS|rare=0|modality=VACCINE'}

## Rule Examples

These examples exercise the operational fallback rules directly. They are intentionally small and deterministic so changes in lookup behavior are easy to spot.

In [7]:
def summarize_lookup(label, snapshot):
    rows = []
    for metric in ("enrollment", "site_count", "patients_per_site", "primary_completion_months", "duration_months"):
        row = opb.lookup_operational_benchmark(snapshot, benchmarks, metric_prefix=metric)
        rows.append({
            "example": label,
            "metric": metric,
            "level": None if row is None else row.get("benchmark_level_used"),
            "key": None if row is None else row.get("benchmark_key"),
            "n": None if row is None else int(row.get(f"{metric}_n")),
            "p50": None if row is None else row.get(f"{metric}_p50"),
            "low_confidence": None if row is None else bool(row.get(f"{metric}_low_confidence_flag")),
        })
    return rows

rule_examples = [
    (
        "invalid indication -> TA fallback",
        {
            "phase": "PHASE3",
            "gbd_cause_id_3_ml": 0,
            "therapeutic_area": "ONCOLOGY",
            "is_rare_disease_ml": 0,
            "therapeutic_modality_ui": "SMALL MOLECULE",
            "primary_duration_months_ml": 9,
        },
    ),
    (
        "invalid TA -> keep indication",
        {
            "phase": "PHASE3",
            "gbd_cause_id_3_ml": 426,
            "therapeutic_area": "UNCLASSIFIED",
            "is_rare_disease_ml": 0,
            "therapeutic_modality_ui": "SMALL MOLECULE",
            "primary_duration_months_ml": 9,
        },
    ),
    (
        "unknown modality -> no modality refinement",
        {
            "phase": "PHASE3",
            "gbd_cause_id_3_ml": 426,
            "therapeutic_area": "ONCOLOGY",
            "is_rare_disease_ml": 0,
            "therapeutic_modality_ui": "UNKNOWN",
            "primary_duration_months_ml": 9,
        },
    ),
    (
        "vaccine Infections -> vaccine refinement",
        {
            "phase": "PHASE3",
            "gbd_cause_id_3_ml": 302,
            "therapeutic_area": "INFECTIONS",
            "is_rare_disease_ml": 0,
            "therapeutic_modality_ui": "VACCINE",
            "primary_duration_months_ml": 9,
        },
    ),
    (
        "non-vaccine Infections -> non-vaccine fallback",
        {
            "phase": "PHASE3",
            "gbd_cause_id_3_ml": 302,
            "therapeutic_area": "INFECTIONS",
            "is_rare_disease_ml": 0,
            "therapeutic_modality_ui": "BIOLOGIC MAB",
            "primary_duration_months_ml": 9,
        },
    ),
]

pd.DataFrame([row for label, snapshot in rule_examples for row in summarize_lookup(label, snapshot)])


,example,metric,level,key,n,p50,low_confidence
0,invalid indication -> TA fallback,enrollment,phase_ta_rare_modality,phase_ta_rare_modality|phase=PHASE3|ta=ONCOLOG...,395,413.00,False
1,invalid indication -> TA fallback,site_count,phase_ta_rare,phase_ta_rare|phase=PHASE3|ta=ONCOLOGY|rare=0,870,78.00,False
2,invalid indication -> TA fallback,patients_per_site,phase_ta_rare_modality,phase_ta_rare_modality|phase=PHASE3|ta=ONCOLOG...,391,5.00,False
3,invalid indication -> TA fallback,primary_completion_months,phase_ta_rare_endpoint_bin,phase_ta_rare_endpoint_bin|phase=PHASE3|ta=ONC...,89,28.55,False
4,invalid indication -> TA fallback,duration_months,phase_ta_rare_endpoint_bin,phase_ta_rare_endpoint_bin|phase=PHASE3|ta=ONC...,88,46.99,False
5,invalid TA -> keep indication,enrollment,phase_indication_rare_modality,phase_indication_rare_modality|phase=PHASE3|in...,60,355.50,False
6,invalid TA -> keep indication,site_count,phase_indication_rare,phase_indication_rare|phase=PHASE3|indication=...,165,78.00,False
7,invalid TA -> keep indication,patients_per_site,phase_indication_rare_modality,phase_indication_rare_modality|phase=PHASE3|in...,60,7.12,False
8,invalid TA -> keep indication,primary_completion_months,phase_endpoint_bin,phase_endpoint_bin|phase=PHASE3|endpoint_bin=6-12,1415,24.02,False
9,invalid TA -> keep indication,duration_months,phase_endpoint_bin,phase_endpoint_bin|phase=PHASE3|endpoint_bin=6-12,1402,28.30,False


## Planned Sites Default Where Enrollment-Coherent Candidate Wins

This example uses a high planned enrollment and low current registry facility-count proxy. The default should choose `planned_enrollment / patients_per_site_p50` because it is larger than the current proxy.

In [8]:
site_win_snapshot = {
    "phase": "PHASE3",
    "gbd_cause_id_3_ml": 302,
    "therapeutic_area": "INFECTIONS",
    "is_rare_disease_ml": 0,
    "therapeutic_modality_ui": "VACCINE",
}

opb.planned_sites_default_from_operational_benchmark(
    site_win_snapshot,
    planned_enrollment=7500,
    current_registry_facility_count_proxy=12,
    overall_status="RECRUITING",
    artifact=benchmarks,
)

{'value': 100,
 'source': 'enrollment_coherent_benchmark_default',
 'site_default_basis': 'enrollment_coherent_benchmark_default',
 'current_registry_facility_count_proxy': 12.0,
 'site_count_benchmark_p50': 14.0,
 'patients_per_site_p50': 75.0,
 'patients_per_site_benchmark_level_used': 'phase_ta_rare_modality',
 'patients_per_site_n': 647,
 'patients_per_site_low_confidence_flag': False,
 'enrollment_coherent_site_candidate': 100.0,
 'operational_benchmark_snapshot_id': 'operational_benchmarks_v1:41bcafd6226cd999:phase_ta_rare_modality|phase=PHASE3|ta=INFECTIONS|rare=0|modality=VACCINE'}

## Validation

The single checker validates schema, fallback behavior, modality/non-vaccine rules, registry/report coverage, site and duration defaulting safety, and model-boundary safeguards. Run it before relying on the artifact after any rebuild.


In [9]:
import json

report = json.loads(REPORT_PATH.read_text())
expected_metrics = [
    "enrollment",
    "site_count",
    "patients_per_site",
    "primary_completion_months",
    "duration_months",
]

for metric in expected_metrics:
    for suffix in ("n", "p25", "p50", "p75", "p90", "low_confidence_flag"):
        assert f"{metric}_{suffix}" in artifact.columns, f"missing {metric}_{suffix}"
    assert report["coverage_qa"][metric]["not_available"] == 0
    assert report["coverage_qa"][metric]["low_confidence_matches"] == 0

artifact_rows = artifact.shape[0]
duration_rows = artifact[artifact["duration_months_n"].fillna(0).gt(0)].shape[0]
endpoint_duration_rows = artifact[
    artifact["benchmark_level_used"].str.contains("endpoint_bin", regex=False)
    & artifact["duration_months_n"].fillna(0).gt(0)
].shape[0]

{
    "artifact_rows": artifact_rows,
    "duration_rows": duration_rows,
    "endpoint_duration_rows": endpoint_duration_rows,
    "completed_actual_total_duration_targets": report["completed_actual_total_duration_targets"],
    "completed_actual_primary_completion_targets": report["completed_actual_primary_completion_targets"],
}


{'artifact_rows': 7406,
 'duration_rows': 4253,
 'endpoint_duration_rows': 3390,
 'completed_actual_total_duration_targets': 20476,
 'completed_actual_primary_completion_targets': 20681}

## Duration Source Priority Examples

These checks exercise direct completed ACTUAL duration, active ESTIMATED duration, stopped-trial lower-bound handling, and primary-completion warning metadata.


In [10]:
duration_examples = {
    "completed_actual": {
        **example_snapshot,
        "overall_status": "COMPLETED",
        "completion_date_type": "ACTUAL",
        "completion_duration_months": 37,
        "primary_completion_date_type": "ACTUAL",
        "primary_completion_duration_months": 20,
    },
    "active_estimated": {
        **example_snapshot,
        "overall_status": "RECRUITING",
        "completion_date_type": "ESTIMATED",
        "completion_duration_months": 42,
        "primary_completion_date_type": "ESTIMATED",
        "primary_completion_duration_months": 24,
    },
    "stopped_actual_floor": {
        **example_snapshot,
        "overall_status": "TERMINATED",
        "completion_date_type": "ACTUAL",
        "completion_duration_months": 50,
        "primary_completion_date_type": "ACTUAL",
        "primary_completion_duration_months": 20,
    },
    "benchmark_default": {
        **example_snapshot,
        "overall_status": "RECRUITING",
        "completion_date_type": None,
        "completion_duration_months": None,
        "primary_completion_date_type": None,
        "primary_completion_duration_months": None,
    },
}

rows = []
for label, snapshot in duration_examples.items():
    total_default = opb.planned_duration_default_from_operational_benchmark(snapshot, artifact=benchmarks)
    duration_row = total_default.get("benchmark_row")
    primary_row = total_default.get("primary_benchmark_row")
    rows.append({
        "example": label,
        "duration_value": total_default.get("value"),
        "duration_source": total_default.get("source"),
        "duration_level": None if duration_row is None else duration_row.get("benchmark_level_used"),
        "duration_n": None if duration_row is None else int(duration_row.get("duration_months_n")),
        "primary_value": total_default.get("planned_primary_completion_months"),
        "primary_source": total_default.get("primary_completion_source"),
        "primary_level": None if primary_row is None else primary_row.get("benchmark_level_used"),
        "primary_n": None if primary_row is None else int(primary_row.get("primary_completion_months_n")),
        "warnings": sorted(set(total_default.get("warnings") or [])),
    })

examples_df = pd.DataFrame(rows)
assert examples_df.loc[examples_df["example"].eq("completed_actual"), "duration_source"].iloc[0] == "final_observed_total_duration"
assert examples_df.loc[examples_df["example"].eq("active_estimated"), "duration_source"].iloc[0] == "estimated_planned_total_duration"
assert examples_df.loc[examples_df["example"].eq("stopped_actual_floor"), "duration_source"].iloc[0] == "benchmark_default_with_floors"
assert examples_df["duration_n"].dropna().ge(50).all()
with_primary = examples_df[examples_df["primary_n"].notna()]
assert with_primary["primary_n"].ge(50).all()
assert (with_primary["primary_level"] == with_primary["duration_level"]).all()
examples_df


,example,duration_value,duration_source,duration_level,duration_n,primary_value,primary_source,primary_level,primary_n,warnings
0,completed_actual,37.00,final_observed_total_duration,phase_ta_rare_endpoint_bin,159,17.82,same_cohort_benchmark,phase_ta_rare_endpoint_bin,160,[]
1,active_estimated,42.00,estimated_planned_total_duration,phase_ta_rare_endpoint_bin,159,17.82,same_cohort_benchmark,phase_ta_rare_endpoint_bin,160,[]
2,stopped_actual_floor,50.00,benchmark_default_with_floors,phase_ta_rare_endpoint_bin,159,17.82,same_cohort_benchmark,phase_ta_rare_endpoint_bin,160,[]
3,benchmark_default,30.03,benchmark_default_with_floors,phase_ta_rare_endpoint_bin,159,17.82,same_cohort_benchmark,phase_ta_rare_endpoint_bin,160,[]


In [11]:
metadata = opb.planned_duration_months_metadata(duration_examples["active_estimated"], artifact=benchmarks)
planned_duration = metadata["planned_duration_months"]
assert planned_duration["support_level"] == "not_evaluated"
assert planned_duration["benchmark_n"] >= 50
if planned_duration["primary_completion_n"] is not None:
    assert planned_duration["primary_completion_n"] >= 50
    assert planned_duration["primary_completion_benchmark_level_used"] == planned_duration["benchmark_level_used"]
planned_duration


{'value': 42.0,
 'source': 'estimated_planned_total_duration',
 'duration_definition': 'start_date_to_completion_date_months',
 'benchmark_level_used': 'phase_ta_rare_endpoint_bin',
 'benchmark_n': 159,
 'benchmark_p25': 15.0,
 'benchmark_p50': 30.03,
 'benchmark_p75': 55.29,
 'benchmark_p90': 80.01,
 'duration_status': 'typical',
 'support_level': 'not_evaluated',
 'supporting_signals': [],
 'conflicting_signals': [],
 'benchmark_snapshot_id': 'operational_benchmarks_v1:41bcafd6226cd999:phase_ta_rare_endpoint_bin|phase=PHASE3|ta=INFECTIONS|rare=0|endpoint_bin=6-12',
 'is_benchmark_stale': False,
 'low_confidence_flag': False,
 'planned_primary_completion_months': 17.82,
 'primary_completion_source': 'same_cohort_benchmark',
 'primary_completion_benchmark_level_used': 'phase_ta_rare_endpoint_bin',
 'primary_completion_n': 160,
 'primary_completion_low_confidence_flag': False,
 'primary_completion_duration_months_context': 24.0,
 'endpoint_duration_months_context': 9.0,
 'actual_total_d

In [12]:
import subprocess

result = subprocess.run(
    [sys.executable, "scripts/check_operational_benchmarks.py"],
    cwd=PROJECT_ROOT,
    check=True,
    capture_output=True,
    text=True,
)
print(result.stdout)


Operational benchmark checks passed.

